In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

# --------------------------------------------------------------
# 2. LOAD DATASET
# --------------------------------------------------------------

# For Google Colab, upload the CSV file first
file_path = "Day12_Used_Car_Preprocessing_Dataset (7).csv"

df = pd.read_csv(file_path)

print("========== DATASET LOADED ==========")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

# --------------------------------------------------------------
# 3. BASIC DATASET INSPECTION
# --------------------------------------------------------------

print("\n========== DATASET INFORMATION ==========")
print(df.info())

print("\n========== STATISTICAL SUMMARY ==========")
print(df.describe())

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

print("\n========== DUPLICATE RECORDS ==========")
print("Number of duplicate rows:", df.duplicated().sum())

# Remove duplicate rows if any
df = df.drop_duplicates().reset_index(drop=True)

print("\nShape after removing duplicates:", df.shape)

# --------------------------------------------------------------
# 4. REMOVE ID COLUMN
# --------------------------------------------------------------

# Car_ID is an identifier and does not provide useful predictive
# information, so it is removed.

if "Car_ID" in df.columns:
    df = df.drop(columns=["Car_ID"])

print("\n========== COLUMNS AFTER REMOVING ID ==========")
print(df.columns.tolist())

# --------------------------------------------------------------
# 5. DEFINE TARGET VARIABLE
# --------------------------------------------------------------

target = "Resale_Price_Lakh"

X = df.drop(columns=[target])
y = df[target]

print("\n========== FEATURES AND TARGET ==========")
print("Features:")
print(X.columns.tolist())

print("\nTarget:", target)

# --------------------------------------------------------------
# 6. IDENTIFY NUMERICAL AND CATEGORICAL COLUMNS
# --------------------------------------------------------------

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\n========== NUMERICAL FEATURES ==========")
print(numerical_features)

print("\n========== CATEGORICAL FEATURES ==========")
print(categorical_features)

# --------------------------------------------------------------
# 7. CHECK OUTLIERS USING IQR
# --------------------------------------------------------------

print("\n========== OUTLIER DETECTION USING IQR ==========")

for col in numerical_features:

    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = X[
        (X[col] < lower_bound) |
        (X[col] > upper_bound)
    ]

    print(
        f"{col}: {len(outliers)} outliers "
        f"(Lower={lower_bound:.2f}, Upper={upper_bound:.2f})"
    )

# --------------------------------------------------------------
# 8. CUSTOM IQR OUTLIER HANDLER
# --------------------------------------------------------------

class IQRClipper(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):

        X = pd.DataFrame(X)

        self.lower_bounds_ = X.quantile(0.25) - (
            1.5 * (X.quantile(0.75) - X.quantile(0.25))
        )

        self.upper_bounds_ = X.quantile(0.75) + (
            1.5 * (X.quantile(0.75) - X.quantile(0.25))
        )

        return self

    def transform(self, X):

        X = pd.DataFrame(X).copy()

        for i in range(X.shape[1]):

            X.iloc[:, i] = X.iloc[:, i].clip(
                lower=self.lower_bounds_.iloc[i],
                upper=self.upper_bounds_.iloc[i]
            )

        return X.values

    # Add get_feature_names_out method
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            # In the context of ColumnTransformer, input_features will be provided.
            # If called directly without input_features, it might not have them.
            raise ValueError("input_features must be provided to get_feature_names_out")
        return input_features


# --------------------------------------------------------------
# 9. CREATE NUMERICAL PREPROCESSING PIPELINE
# --------------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("outlier_handler", IQRClipper()),
        ("scaler", StandardScaler())
    ]
)

# --------------------------------------------------------------
# 10. CREATE CATEGORICAL PREPROCESSING PIPELINE
# --------------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

# --------------------------------------------------------------
# 11. COMBINE PREPROCESSING
# --------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

# --------------------------------------------------------------
# 12. TRAIN-TEST SPLIT
# --------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\n========== TRAIN TEST SPLIT ==========")
print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

# --------------------------------------------------------------
# 13. FIT PREPROCESSING ONLY ON TRAINING DATA
# --------------------------------------------------------------

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("\n========== PROCESSED DATA ==========")
print("X_train processed shape:", X_train_processed.shape)
print("X_test processed shape :", X_test_processed.shape)

# --------------------------------------------------------------
# 14. GET FEATURE NAMES
# --------------------------------------------------------------

feature_names = preprocessor.get_feature_names_out()

# Remove pipeline prefixes for cleaner column names
feature_names = [
    name.replace("num__", "")
        .replace("cat__", "")
        .replace("encoder__", "")
    for name in feature_names
]

# --------------------------------------------------------------
# 15. CONVERT PROCESSED DATA TO DATAFRAME
# --------------------------------------------------------------

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

# --------------------------------------------------------------
# 16. ADD TARGET VARIABLE
# --------------------------------------------------------------

train_processed = X_train_processed_df.copy()
train_processed[target] = y_train.values

test_processed = X_test_processed_df.copy()
test_processed[target] = y_test.values

# --------------------------------------------------------------
# 17. VERIFY PROCESSED DATA
# --------------------------------------------------------------

print("\n========== PROCESSED TRAINING DATA ==========")
print(train_processed.head())

print("\n========== PROCESSED TESTING DATA ==========")
print(test_processed.head())

print("\nProcessed training shape:", train_processed.shape)
print("Processed testing shape :", test_processed.shape)

print("\n========== CHECK MISSING VALUES ==========")
print(
    "Training missing values:",
    train_processed.isnull().sum().sum()
)

print(
    "Testing missing values:",
    test_processed.isnull().sum().sum()
)

# --------------------------------------------------------------
# 18. CHECK SCALING
# --------------------------------------------------------------

print("\n========== SCALED NUMERICAL FEATURES ==========")

print(
    train_processed[
        [col for col in feature_names
         if col in numerical_features]
    ].describe().round(3)
)

# --------------------------------------------------------------
# 19. SAVE PREPROCESSED DATASETS
# --------------------------------------------------------------

train_processed.to_csv(
    "Used_Car_Preprocessed_Train.csv",
    index=False
)

test_processed.to_csv(
    "Used_Car_Preprocessed_Test.csv",
    index=False
)

# Save complete processed dataset
processed_full = pd.concat(
    [train_processed, test_processed]
).sort_index()

processed_full.to_csv(
    "Used_Car_Preprocessed_Dataset.csv",
    index=False
)

print("\n========== FILES SAVED ==========")
print("1. Used_Car_Preprocessed_Train.csv")
print("2. Used_Car_Preprocessed_Test.csv")
print("3. Used_Car_Preprocessed_Dataset.csv")

# --------------------------------------------------------------
# 20. FINAL VERIFICATION
# --------------------------------------------------------------

print("\n========== FINAL VERIFICATION ==========")

print("Original dataset shape:", df.shape)

print(
    "Processed complete dataset shape:",
    processed_full.shape
)

print(
    "Training dataset shape:",
    train_processed.shape
)

print(
    "Testing dataset shape:",
    test_processed.shape
)

print(
    "\nMissing values in final dataset:",
    processed_full.isnull().sum().sum()
)

print(
    "Duplicate rows in final dataset:",
    processed_full.duplicated().sum()
)

print("\n========== PREPROCESSING COMPLETED SUCCESSFULLY ==========")


========== DATASET LOADED ==========
Shape: (320, 15)

First 5 rows:
    Car_ID       Brand  Year  Mileage_Km  Engine_CC  Power_BHP Fuel_Type  \
0  CAR0001       Skoda  2021       69708       1152      128.8    Diesel   
1  CAR0002      Toyota  2020       88881        903      146.5    Diesel   
2  CAR0003  Volkswagen  2021       43646       1446      185.9    Diesel   
3  CAR0004        Tata  2019       70847       2069      148.8    Petrol   
4  CAR0005        Tata  2016      101228       1657      206.0    Petrol   

  Transmission        City Seller_Type  Condition  Previous_Owners  \
0       Manual     Lucknow  Individual       Good                1   
1    Automatic  Chandigarh  Individual       Good                1   
2    Automatic   Hyderabad  Individual  Very Good                2   
3       Manual     Lucknow  Individual  Excellent                3   
4    Automatic   Ahmedabad      Dealer  Very Good                2   

   Accidents_Reported  Service_Score  Resale_Price_La